![iceberg-logo](https://www.apache.org/logos/res/iceberg/iceberg.png)

### [Docker, Spark, and Iceberg: The Fastest Way to Try Iceberg!](https://tabular.io/blog/docker-spark-and-iceberg/)

### SQL Authoring Note (Kotlin Kernel)

Known limitation (Kotlin kernel): JupyterLab does not apply SQL syntax highlighting inside Kotlin triple-quoted strings (`"""..."""`).

What works in this notebook:
- Use the toolbar **Format SQL** action for query formatting.
- Keep SQL in a dedicated variable and execute with `spark.sql(query)`.

```kotlin
val query = """
SELECT *
FROM nyc.taxis
""".trimIndent()

spark.sql(query).show(100, 0, false)
```


In [ ]:
import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder().appName("Jupyter").getOrCreate()

spark.version()

# Write support

This notebook demonstrates writing to Iceberg tables using Kotlin + Spark SQL. First, connect to the [catalog](https://iceberg.apache.org/concepts/catalog/#iceberg-catalogs), the place where tables are being tracked.

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS default")

# Create an Iceberg table

Next create the Iceberg table directly from the `Spark DataFrame`.

In [ ]:
val table_name = "default.commits"

spark.sql("DROP TABLE IF EXISTS $table_name")
spark.sql("""
CREATE TABLE $table_name (
    id BIGINT,
    name STRING,
    state STRING,
    additions BIGINT,
    deletes BIGINT
)
USING iceberg
""".trimIndent())

spark.table(table_name)

# Loading data using Arrow

Create an example Spark DataFrame table that mimics data from the GitHub API.

In [ ]:
spark.sql("""
INSERT INTO $table_name VALUES
    (123, 'Fix bug', 'Open', 22, 10),
    (234, 'Add VariantType', 'Open', 29123, 302),
    (345, 'Add commit retries', 'Open', 22, 10)
""".trimIndent())

spark.table(table_name).orderBy("id").show(100, 0, false)

# Write the data

Let's append the data to the table:

In [ ]:
val commit_count = spark.table(table_name).count()
check(commit_count == 3L) { "Expected 3 rows, found $commit_count" }

spark.table(table_name).orderBy("id").show(100, 0, false)

In [ ]:
spark.sql("""
SELECT snapshot_id, committed_at, operation
FROM $table_name.snapshots
ORDER BY committed_at
""".trimIndent()).show(false)

# Add moar data

In [ ]:
spark.sql("""
INSERT INTO $table_name VALUES
    (456, 'Add NanosecondTimestamps', 'Merged', 2392, 8),
    (567, 'Add documentation around filters', 'Open', 7543, 3)
""".trimIndent())

spark.table(table_name).orderBy("id").show(100, 0, false)

In [ ]:
spark.sql("""
SELECT snapshot_id, committed_at, operation
FROM $table_name.snapshots
ORDER BY committed_at
""".trimIndent()).show(false)

# Upsert new data

In [ ]:
spark.sql("""
MERGE INTO $table_name AS target
USING (
    SELECT *
    FROM VALUES
        (456, 'Add NanosecondTimestamps', 'Merged', 2392, 8),
        (567, 'Add documentation around filters', 'Merged', 9238, 22)
        AS updates(id, name, state, additions, deletes)
) AS source
ON target.id = source.id
WHEN MATCHED THEN UPDATE SET
    target.name = source.name,
    target.state = source.state,
    target.additions = source.additions,
    target.deletes = source.deletes
WHEN NOT MATCHED THEN INSERT (id, name, state, additions, deletes)
VALUES (source.id, source.name, source.state, source.additions, source.deletes)
""".trimIndent())

spark.table(table_name).orderBy("id").show(100, 0, false)